# End-to-end exact deduplication

GPU accelerated exact deduplication. Two documents are exact duplicates when their text is identical. For more information about exact deduplication in NeMo Curator, refer to the [Exact Duplicate Removal](https://docs.nvidia.com/nemo/curator/latest/curate-text/process-data/deduplication/exact) documentation page.

The tutorial here shows how to run exact deduplication on text data by executing 2 end to end workflows:

1. `ExactDeduplicationWorkflow` (GPU) reads the dataset, hashes the text of every document with MD5, groups documents with the same hash, keeps one document per group and writes the IDs of the rest to a removal list.
2. `TextDuplicatesRemovalWorkflow` (CPU) reads the dataset again and drops the documents in the removal list.

In [1]:
import os

# Silence Curator logs via Loguru
os.environ["LOGURU_LEVEL"] = "ERROR"

import pandas as pd

input_dataset_path = "./input"  # Path to input dataset
exact_output_dir = "./exact_outputs"  # Path to store all exact dedup outputs; use an empty directory for each new run
deduplicated_output_path = os.path.join(exact_output_dir, "exact_deduped_dataset")

input_filetype = (
    "parquet"  # this can be either of jsonl or parquet (you'll need to change how input data is generated)
)
# Note: It's important that this is constant across identification and removal.
# Removal finds the IDs of each group of input files by the files in it, so both workflows must build the same groups.
input_blocksize = "512MB"
output_filetype = "parquet"  # this can be either of jsonl or parquet

storage_options = None  # Optional additional cloud I/O args to pass into Pandas/cuDF during I/O operations.
io_kwargs = {"storage_options": storage_options} if storage_options is not None else None

### Downloading and saving a sample dataset

We download and save the [Tinystories](https://huggingface.co/datasets/roneneldan/TinyStories) dataset to the specified `input_dataset_path` above. This step can be skipped if running on a different dataset that's already present in the input_dataset_path.

In [2]:
from nemo_curator.utils.file_utils import get_all_file_paths_under

if len(get_all_file_paths_under(input_dataset_path, storage_options=storage_options)) == 0:
    import os
    import uuid

    from datasets import load_dataset

    input_df = load_dataset("roneneldan/TinyStories", split="train").to_pandas()
    num_rows_per_file = 10_000

    os.makedirs(input_dataset_path, exist_ok=True)

    for i, start_idx in enumerate(range(0, len(input_df), num_rows_per_file)):
        if i % 50 == 0:
            print(f"Processing file {i}")
        end_idx = min(len(input_df), start_idx + num_rows_per_file)
        subset_df = input_df.iloc[start_idx:end_idx].copy()
        subset_df["id"] = [str(uuid.uuid4()) for _ in range(len(subset_df))]
        subset_df.to_parquet(
            os.path.join(input_dataset_path, f"part_{i}.parquet"), index=False, storage_options=storage_options
        )

    print(f"Created {i + 1} files")

Processing file 0


Processing file 50


Processing file 100


Processing file 150


Processing file 200


Created 212 files


## Running exact deduplication

### General Notes
#### How duplicates are found
1. The `text_field` of every document is hashed with MD5 on the GPU. Documents with the same hash are exact duplicates.
2. Set `normalize_text=True` to lowercase the text, collapse runs of whitespace and trim it before hashing. Documents that differ only in case or spacing then count as duplicates too.
3. One document from each group of duplicates is kept and the IDs of the rest are written to `ExactDuplicateIds`.

#### ID Generation
1. The ID generation process requires a Ray cluster to be started before running the workflow either from the CLI or by using the `RayClient` API in Curator.
2. The ID Generator gives each row a unique increasing integer ID, based on the order files are read.
3. The workflow also saves an `exact_id_generator.json` which maintains a mapping of input file groups to ID ranges for that group.
4. During removal, reading the same file groups gives the same integer IDs. This is why `input_blocksize` must be identical for identification and removal: a different blocksize builds different file groups, and their IDs can't be found in `exact_id_generator.json`.

#### Removal
1. `ExactDeduplicationWorkflow(perform_removal=True)` is not implemented yet, so removal is a separate step that uses `TextDuplicatesRemovalWorkflow`, the same workflow used for fuzzy deduplication.
2. The removal workflow is CPU only and can be run on machines that don't have GPUs. As in the fuzzy deduplication tutorial, we pass the `RayDataExecutor`, which showed better results during duplicate removal.

In [3]:
import time

import torch

from nemo_curator.backends.ray_data import RayDataExecutor
from nemo_curator.core.client import RayClient

NUM_GPUS = 1

if torch.cuda.device_count() < NUM_GPUS:
    error_msg = "The number of GPUs on this machine are lesser than the default this tutorial was tested with, please update `num_gpus` passed into `RayClient`"
    raise ValueError(error_msg)

client = RayClient(num_gpus=NUM_GPUS)  # change as needed
client.start()

In [4]:
from nemo_curator.stages.deduplication.exact.workflow import ExactDeduplicationWorkflow

# All workflows support passing in different kwargs and storage_options for the read and output datasets
# We use a common one here for simplicity

identification_workflow = ExactDeduplicationWorkflow(
    output_path=exact_output_dir,
    input_path=input_dataset_path,
    input_filetype=input_filetype,
    input_blocksize=input_blocksize,
    text_field="text",
    normalize_text=False,  # set to True to ignore differences in case and whitespace
    assign_id=True,
    read_kwargs=io_kwargs,
    write_kwargs=io_kwargs,
)

In [5]:
st = time.time()
identification_result = identification_workflow.run()
print(f"Identification workflow took: {(time.time() - st):.2f}s")
print(f"Number of exact duplicates found: {identification_result.metadata['num_duplicates']}")

Identification workflow took: 34.40s
Number of exact duplicates found: 320470


In [6]:
from nemo_curator.stages.deduplication.id_generator import CURATOR_DEDUP_ID_STR
from nemo_curator.stages.text.deduplication import TextDuplicatesRemovalWorkflow

removal_workflow = TextDuplicatesRemovalWorkflow(
    input_path=input_dataset_path,  # Must be identical to the path used during identification
    ids_to_remove_path=os.path.join(exact_output_dir, "ExactDuplicateIds"),
    output_path=deduplicated_output_path,
    input_filetype=input_filetype,
    input_blocksize=input_blocksize,  # This must be identical to the blocksize used during identification
    duplicate_id_field=CURATOR_DEDUP_ID_STR,
    id_generator_path=os.path.join(exact_output_dir, "exact_id_generator.json"),
    output_filetype=output_filetype,
    input_kwargs=io_kwargs,  # read_kwargs for input dataset
    duplicate_id_read_kwargs=io_kwargs,  # read_kwargs for removal_id's generated by the exact workflow
    id_generator_storage_options=storage_options,
    output_kwargs=io_kwargs,
)

In [7]:
st = time.time()
_ = removal_workflow.run(executor=RayDataExecutor())
print(f"Removal workflow took: {(time.time() - st):.2f}s")

Removal workflow took: 39.60s


### Looking at the results

#### ExactDuplicateIds (list of duplicate documents to remove)
1. `_curator_dedup_id` - ID of documents in the removal list

In [8]:
duplicate_ids_path = os.path.join(exact_output_dir, "ExactDuplicateIds")
duplicates_df = pd.read_parquet(duplicate_ids_path, storage_options=storage_options)
display(duplicates_df.head())

print(f"Number of duplicate documents found for removal: {len(duplicates_df)}")

,_curator_dedup_id
0,11758
1,11759
2,11760
3,11761
4,11762


Number of duplicate documents found for removal: 320470


#### Checking the results against pandas

Exact duplicates can also be counted directly with pandas, which gives an independent check of both workflows. This only works for small datasets that fit in memory.

In [9]:
input_df = pd.read_parquet(input_dataset_path, columns=["text"], storage_options=storage_options)
deduped_df = pd.read_parquet(deduplicated_output_path, columns=["text"], storage_options=storage_options)

print(f"Number of documents in the input dataset: {len(input_df)}")
print(f"Number of documents in the deduplicated dataset: {len(deduped_df)}")

# pandas treats a missing text differently from an empty one, so the check below needs no missing texts
assert input_df.text.notna().all()  # noqa: S101
# Every document whose text already appeared earlier is in the removal list
assert len(duplicates_df) == input_df.text.duplicated().sum()  # noqa: S101
# Removal drops exactly the documents in the removal list
assert len(deduped_df) == len(input_df) - len(duplicates_df)  # noqa: S101
# No exact duplicates are left
assert deduped_df.text.duplicated().sum() == 0  # noqa: S101

Number of documents in the input dataset: 2119719
Number of documents in the deduplicated dataset: 1799249


#### Checking that one document per duplicate group is kept

In [10]:
duplicated_texts = input_df.text[input_df.text.duplicated(keep=False)].unique()
kept_docs = deduped_df.text[deduped_df.text.isin(duplicated_texts)]

print(f"Number of distinct texts with more than one copy in the input: {len(duplicated_texts)}")
print(f"Number of copies of those texts left after removal: {len(kept_docs)}")
assert len(kept_docs) == len(duplicated_texts)  # noqa: S101

Number of distinct texts with more than one copy in the input: 320042
Number of copies of those texts left after removal: 320042


#### Looking at examples of duplicate documents

In [11]:
duplicate_group_sizes = input_df.groupby("text").size().sort_values(ascending=False)
duplicate_group_sizes = duplicate_group_sizes[duplicate_group_sizes > 1]

# How many duplicate groups there are of each size
display(duplicate_group_sizes.value_counts().sort_index().rename_axis("group_size").rename("num_groups").to_frame())

# The largest groups, with the text shortened
largest_groups = duplicate_group_sizes.head().rename("group_size").reset_index()
largest_groups["text"] = largest_groups["text"].str.slice(0, 100)
display(largest_groups)

,num_groups
group_size,
2,319841
3,200
230,1


,text,group_size
0,,230
1,Lily and Tom were playing with their dolls in ...,3
2,"Once upon a time, there was a happy little gir...",3
3,Lily and Ben were twins who liked to play outs...,3
4,Sam and Tom are friends. They like to play nea...,3


The largest group of duplicates in this dataset, 230 documents, is all documents with empty text. The other groups are stories that appear 2 or 3 times.

In [12]:
client.stop()

### Conclusion
We found and removed 320,470 exact duplicate documents from a dataset of ~2.1 million rows (2,119,719 documents before removal, 1,799,249 after).